# Практика · U-Net

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі сцени зошит малює формулами, ваги не завантажуються.
> Досить `torch`, `numpy` і `matplotlib`.

> ⏱ Зошит навчає **55 мереж**: вісімнадцять конфігурацій по три зерна кожна плюс
> одна окрема для розбору активацій. Заміряно: **близько 10 хвилин** на чотирьох
> ядрах без відеокарти, в один потік — із них чистого процесорного часу 5.5
> хвилини, решта припадає на очікування, коли машина зайнята чимось іще.
> Час тут не витрата, а сам предмет: різниця, менша за розкид по зернах,
> різницею не є, а розкид без трьох зерен не порахуєш.

Що зробимо:

1. згенеруємо сцени 64 × 64 з масками й порахуємо, скільки в них фону;
2. напишемо **свою смугу межі** й звіримо її з наївною реалізацією в лоб;
3. звіримо свій `mIoU` з `sklearn`;
4. поміряємо **стелю роздільності**: скільки лишається від маски, стиснутої в 2, 4, 8, 16 разів;
5. зберемо U-Net із керованою глибиною й керованими скіпами;
6. знайдемо **порогову глибину** — з якої скіпи починають вирішувати;
7. порівняємо **конкатенацію з додаванням**;
8. вимкнемо скіпи по одному й побачимо, який із них важить;
9. подивимось на **активації**, які йдуть скіпом, проти тих, що приходять знизу;
10. перевіримо, чи потрібна **симетрія** декодера;
11. порівняємо `padding='same'` із **дзеркальним доповненням**;
12. поміряємо IoU **кільця** зі стінкою 1, 2 і 3 пікселі — і побачимо, заради чого U-Net узагалі придумали.

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми додаються в іншому порядку
torch.set_num_threads(1)

NOTEBOOK_STARTED = time.perf_counter()

print("torch    ", torch.__version__)
print("numpy    ", np.__version__)
print("потоків  ", torch.get_num_threads())

## 1 · Сцени з масками

Полотно 64 × 64, від одного до трьох предметів трьох класів, шум зі стандартним
відхиленням 0.12. Маска — той самий масив, що в блоці детекції давав рамку, тільки
тепер ми беремо його цілком: `seg[маска] = номер класу + 1`. Нуль лишається фону.

Це рідкісна розкіш: розмітка істинна **за побудовою**, до пікселя. На справжніх
даних маска коштує в десятки разів дорожче за рамку — тема 02 рахувала цю ціну.
Наш зошит показує механізм, а не вартість проєкту.

In [ ]:
SIZE = 64
CLASS_NAMES = ["фон", "коло", "квадрат", "трикутник"]


def shape_mask(kind, center_x, center_y, radius, ring_wall=2):
    """Маска однієї фігури на полотні 64 × 64."""
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                    # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                    # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    if kind == 2:                                    # трикутник: ширина росте згори вниз
        return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
                & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))
    # kind == 3 — кільце: коло мінус менше коло, стінка завтовшки ring_wall
    dist2 = (xs - center_x) ** 2 + (ys - center_y) ** 2
    inner = max(1, radius - ring_wall)
    return (dist2 <= radius * radius) & (dist2 > inner * inner)


def make_scene(rng, kinds=(0, 1, 2), ring_wall=2, noise=0.12):
    """Одна сцена: картинка, маска класів і карта товщини стінки кілець."""
    image = np.zeros((SIZE, SIZE), np.float32)
    seg = np.zeros((SIZE, SIZE), np.int64)
    walls = np.zeros((SIZE, SIZE), np.int64)     # 0 скрізь, крім пікселів кільця
    placed = []
    for _ in range(int(rng.integers(1, 4))):
        for _attempt in range(40):
            radius = int(rng.integers(6, 11))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            which = int(rng.integers(0, len(kinds)))
            # у змішаному наборі кожне кільце дістає свою товщину стінки —
            # так одна навчена мережа міряється одразу на всіх трьох товщинах
            this_wall = ring_wall
            if kinds[which] == 3 and ring_wall == "mixed":
                this_wall = int(rng.integers(1, 4))
            mask = shape_mask(kinds[which], center_x, center_y, radius, this_wall)
            ys, xs = np.nonzero(mask)
            if len(ys) == 0:
                continue
            box = [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]

            # не даємо предметам злипатись більше ніж на 45 % площі нового
            too_close = False
            for previous in placed:
                overlap_w = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                overlap_h = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if overlap_w * overlap_h > 0.45 * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close:
                continue

            image[mask] = 1.0
            seg[mask] = which + 1                    # 0 лишається фоном
            if kinds[which] == 3:
                walls[mask] = this_wall
            placed.append(box)
            break
    image = image + rng.normal(0, noise, image.shape).astype(np.float32)
    return image, seg, walls


def make_dataset(how_many, seed, kinds=(0, 1, 2), ring_wall=2):
    """Набір сцен із фіксованим зерном — числа повторюються в кожного читача."""
    rng = np.random.default_rng(seed)
    images, masks, walls = [], [], []
    for _ in range(how_many):
        image, seg, wall = make_scene(rng, kinds, ring_wall)
        images.append(image)
        masks.append(seg)
        walls.append(wall)
    return (torch.from_numpy(np.stack(images)).unsqueeze(1),
            torch.from_numpy(np.stack(masks)),
            torch.from_numpy(np.stack(walls)))


train_images, train_shapes, _ = make_dataset(256, seed=42)
val_images, val_shapes, _ = make_dataset(96, seed=7)

print("навчальних сцен", len(train_images), " перевірних", len(val_images))
print("розмір батча зображень", tuple(train_images.shape))
print()
for class_index in range(4):
    share = (val_shapes == class_index).float().mean().item()
    print("%-10s %6.2f %% пікселів" % (CLASS_NAMES[class_index], 100 * share))

# у цій темі нас цікавить, ДЕ проходить межа, а не яка це фігура, тож три
# предметні класи зливаються в один: «предмет». Розрізняти коло від квадрата
# вчила тема 29; тут це тільки додавало б до розриву плутанину між класами
train_masks = (train_shapes > 0).long()
val_masks = (val_shapes > 0).long()
print()
print("для навчання лишаємо два класи: фон %.2f %%, предмет %.2f %%"
      % (100 * (val_masks == 0).float().mean().item(),
         100 * (val_masks == 1).float().mean().item()))

Фон займає майже девʼять десятих полотна. Модель, яка зафарбує все фоном, дістане
під 90 % правильних пікселів і `IoU` предмета рівно нуль — саме тому середня
точність по пікселях у сегментації нічого не варта. Це те саме, що тема 29
показала на своїх сценах.

**Чому два класи, а не чотири.** Тема 29 вчила мережу розрізняти коло, квадрат і
трикутник. Ця тема — про інше: про те, **де** проходить межа предмета й скільки
від неї лишається після стискання. Якби ми лишили три предметні класи, у розрив
між мережами домішувалась би ще й плутанина «коло проти квадрата», яка до
роздільності стосунку не має. Тому предметні класи зливаємо в один. Через це наші
числа не порівнюються напряму з числами теми 29 — там `mIoU` рахувався по
чотирьох класах, тут по двох.

In [ ]:
plt.figure(figsize=(9, 3.2))
for position in range(3):
    plt.subplot(2, 3, position + 1)
    plt.imshow(val_images[position, 0], cmap="gray")
    plt.axis("off")
    plt.title("сцена %d" % position, fontsize=9)
    plt.subplot(2, 3, position + 4)
    plt.imshow(val_shapes[position], cmap="viridis", vmin=0, vmax=3)
    plt.axis("off")
    plt.title("маска", fontsize=9)
plt.tight_layout()
plt.show()
print("угорі — вхід мережі, унизу — еталонна розмітка")

## 2 · Смуга межі власноруч

Середній `mIoU` рахується по всіх пікселях, а майже всі пікселі предмета лежать
глибоко всередині нього, далеко від краю. Тому метрика, усереднена по всьому
полотну, майже не помічає, чи акуратний край. Нам потрібна метрика **саме на межі**.

Смуга межі — це пікселі, в околі яких зустрічаються **різні** класи. Найпростіший
спосіб її знайти: для кожного пікселя подивитись на квадратний окіл зі стороною
`2w + 1` і перевірити, чи збігаються там максимум і мінімум номера класу. Якщо
ні — окіл накриває кордон, а піксель належить смузі.

Максимум околу дає `max_pool2d`. Мінімум околу — той самий `max_pool2d`, але від
**мінус** маски й із мінусом назад: мінімум чисел дорівнює мінусу максимуму
їхніх протилежних значень.

In [ ]:
def boundary_band(masks, w=2):
    """Смуга завширшки w уздовж будь-якої межі між класами."""
    as_float = masks.unsqueeze(1).float()
    # вікно 2w+1 з кроком 1 і доповненням w — розмір карти не змінюється
    biggest = F.max_pool2d(as_float, 2 * w + 1, 1, w)
    smallest = -F.max_pool2d(-as_float, 2 * w + 1, 1, w)
    return (biggest != smallest).squeeze(1)          # окіл накриває кордон


def boundary_band_naive(masks, w=2):
    """Те саме в лоб, циклами — щоб було з чим звірити швидку версію."""
    result = np.zeros(masks.shape, bool)
    data = masks.numpy()
    for n in range(data.shape[0]):
        for row in range(SIZE):
            for col in range(SIZE):
                r0, r1 = max(0, row - w), min(SIZE, row + w + 1)
                c0, c1 = max(0, col - w), min(SIZE, col + w + 1)
                window = data[n, r0:r1, c0:c1]
                result[n, row, col] = window.max() != window.min()
    return torch.from_numpy(result)


fast = boundary_band(val_masks[:8], 2)
slow = boundary_band_naive(val_masks[:8], 2)

assert torch.equal(fast, slow), "швидка смуга розійшлася з наївною!"
print("✅ смуга через max_pool2d збігається з реалізацією в лоб на всіх 32 768 пікселях")

band = boundary_band(val_masks, 2)
print("смуга межі накриває %.2f %% усіх пікселів" % (100 * band.float().mean().item()))

Смуга займає приблизно восьму частину полотна. Далі кожну модель ми міряємо
двічі: `mIoU` по всіх пікселях і `mIoU` **тільки на цій смузі**.

## 3 · mIoU власноруч і бібліотечний

`IoU` одного класу — це кількість пікселів, де ми і розмітка **обидва** сказали
цей клас, поділена на кількість пікселів, де це сказав **хоч хтось**. `mIoU` —
середнє таких значень по класах; у нас класів два, фон і предмет. Класи, яких
немає ні в розмітці, ні в передбаченні, до середнього не входять: ділити нуль на
нуль немає сенсу.

Звіримо з `sklearn.metrics.jaccard_score(average='macro')` — це та сама величина
під іншою назвою (індекс Жаккара і є IoU).

In [ ]:
from sklearn.metrics import jaccard_score


def class_iou(predicted, truth, class_index):
    """IoU одного класу."""
    p = predicted == class_index
    t = truth == class_index
    union = (p | t).sum().item()
    if union == 0:
        return float("nan")
    return (p & t).sum().item() / union


def mean_iou(predicted, truth, n_classes=2, only_where=None):
    """Середній IoU по класах. only_where обмежує підрахунок смугою межі."""
    if only_where is not None:
        predicted = predicted[only_where]
        truth = truth[only_where]
    values = []
    for class_index in range(n_classes):
        value = class_iou(predicted, truth, class_index)
        if value == value:                           # not nan
            values.append(value)
    return float(np.mean(values))


def pixel_accuracy(predicted, truth, only_where=None):
    if only_where is not None:
        predicted = predicted[only_where]
        truth = truth[only_where]
    return (predicted == truth).float().mean().item()


# зіпсуємо еталонну маску навмисне, щоб було що міряти
rng = np.random.default_rng(0)
spoiled = val_masks.clone()
noise_positions = torch.from_numpy(rng.random(val_masks.shape) < 0.05)
spoiled[noise_positions] = torch.randint(0, 2, (int(noise_positions.sum()),))

ours = mean_iou(spoiled, val_masks)
theirs = jaccard_score(val_masks.numpy().ravel(), spoiled.numpy().ravel(),
                       average="macro", labels=[0, 1])

print("наш mIoU        %.6f" % ours)
print("sklearn         %.6f" % theirs)
assert abs(ours - theirs) < 1e-9, "наш mIoU розійшовся з бібліотечним!"
print("✅ збігається")

## 4 · Стеля роздільності: що взагалі можна відновити з карти рівня k

Перш ніж навчати мережі, поміряємо річ, для якої навчання не потрібне зовсім.

Уяви, що декодер ідеальний: він точно знає, який клас переважає в кожній клітинці
стиснутої карти, і не помиляється ні на йоту. Що він може відновити? Рівно те, що
лишилося в карті. Стиснемо еталонну маску в 2, 4, 8, 16 разів (беручи в кожній
клітинці клас, який займає в ній найбільшу площу), розтягнемо назад до 64 × 64 і
поміряємо `mIoU` цієї реконструкції проти оригіналу.

Це **стеля**: жодна мережа, яка бачить лише карту цього рівня, не зробить краще.

In [ ]:
print("рівень  стиснення  карта     mIoU     на межі  точність межі")
resolution_ceiling = []
for level in range(5):
    step = 2 ** level
    if step == 1:
        restored = val_masks.clone()
    else:
        one_hot = F.one_hot(val_masks, 2).permute(0, 3, 1, 2).float()
        # середнє по клітинці = частка кожного класу в ній
        shares = F.avg_pool2d(one_hot, step)
        stretched = F.interpolate(shares, scale_factor=step, mode="nearest")
        restored = stretched.argmax(1)
    row = (level, step, 64 // step,
           mean_iou(restored, val_masks),
           mean_iou(restored, val_masks, only_where=band),
           pixel_accuracy(restored, val_masks, only_where=band))
    resolution_ceiling.append(row)
    print("  %d       %2d ×     %2d×%-2d   %.4f   %.4f   %.4f"
          % (row[0], row[1], row[2], row[2], row[3], row[4], row[5]))

Ось перша половина відповіді на питання «навіщо взагалі скіпи». Карта, стиснута
у вісім разів, зберігає далеко не все — і це стеля для **ідеального** декодера,
який ще й нічого не плутає. Друга половина відповіді буде в розділі 6: виявиться,
що на малих глибинах мережа підбирається до цієї стелі й без сторонньої допомоги,
а з певної глибини — перестає.

## 5 · U-Net із керованою глибиною

Збираємо одну мережу, у якої вмикаються й вимикаються всі речі, які ми хочемо
поміряти: глибина стискання, наявність кожного окремого скіпа, спосіб зшивання
(конкатенація чи додавання), ширина декодера й режим доповнення на краях.

In [ ]:
def conv_block(in_channels, out_channels, pad_mode="zeros"):
    """Дві згортки 3 × 3 поспіль — стандартна цеглинка U-Net."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, padding_mode=pad_mode),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, padding_mode=pad_mode),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """Енкодер-декодер зі скіпами, які можна вимикати по одному.

    depth      скільки разів стискаємо (3 → карта дна вчетверо менша за 8 разів)
    use_skip   список довжини depth: індекс 0 — найдрібніший рівень, найвищої
               роздільності; True означає «скіп увімкнено»
    merge      'concat' — як в U-Net; 'add' — як залишок у ResNet (тема 14)
    dec_scale  ширина декодера відносно енкодера: 1.0 — симетрія, 0.5 — удвічі тонший
    """

    def __init__(self, depth=3, base=6, use_skip=True, merge="concat",
                 dec_scale=1.0, pad_mode="zeros", n_classes=2):
        super().__init__()
        self.depth = depth
        self.merge = merge
        if use_skip is True:
            use_skip = [True] * depth
        if use_skip is False:
            use_skip = [False] * depth
        self.use_skip = list(use_skip)

        enc_ch = [base * (2 ** i) for i in range(depth + 1)]
        self.enc = nn.ModuleList([conv_block(1, enc_ch[0], pad_mode)])
        for i in range(depth):
            self.enc.append(conv_block(enc_ch[i], enc_ch[i + 1], pad_mode))

        dec_ch = [max(4, int(round(c * dec_scale))) for c in enc_ch]
        dec_ch[depth] = enc_ch[depth]                # дно спільне з енкодером
        self.up, self.dec, self.proj = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for i in range(depth - 1, -1, -1):
            self.up.append(nn.ConvTranspose2d(dec_ch[i + 1], dec_ch[i], 2, 2))
            if self.use_skip[i] and merge == "concat":
                after_merge = dec_ch[i] + enc_ch[i]  # канали складаються
            else:
                after_merge = dec_ch[i]
            self.dec.append(conv_block(after_merge, dec_ch[i], pad_mode))
            # для додавання скіп треба спершу звузити до ширини декодера
            if merge == "add" and self.use_skip[i] and enc_ch[i] != dec_ch[i]:
                self.proj.append(nn.Conv2d(enc_ch[i], dec_ch[i], 1))
            else:
                self.proj.append(nn.Identity())
        self.head = nn.Conv2d(dec_ch[0], n_classes, 1)
        # типова ініціалізація torch для згорток розрахована на інші нелінійності;
        # з нею частина зерен просто не виходила з поганого початкового наближення
        for layer in self.modules():
            if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(layer.weight, mode="fan_out", nonlinearity="relu")
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

    def forward(self, x, return_feats=False):
        saved = []
        x = self.enc[0](x)
        saved.append(x)
        for i in range(self.depth):
            x = F.max_pool2d(x, 2)                   # тут і губиться роздільність
            x = self.enc[i + 1](x)
            saved.append(x)
        bottom = x
        from_below = []
        for step, i in enumerate(range(self.depth - 1, -1, -1)):
            x = self.up[step](x)                     # роздільність назад
            from_below.append(x)
            if self.use_skip[i]:
                skip = self.proj[step](saved[i])
                x = torch.cat([x, skip], 1) if self.merge == "concat" else x + skip
            x = self.dec[step](x)
        out = self.head(x)
        if return_feats:
            return out, saved, from_below, bottom
        return out


def count_params(model):
    return sum(p.numel() for p in model.parameters())


demo = UNet(depth=3)
print("U-Net глибини 3, базова ширина 6 каналів:", count_params(demo), "параметрів")
print("без скіпів:", count_params(UNet(depth=3, use_skip=False)), "параметрів")
print()
with torch.no_grad():
    _, saved, from_below, bottom = demo(val_images[:1], return_feats=True)
for level, tensor in enumerate(saved):
    print("енкодер рівень %d: %s" % (level, tuple(tensor.shape)))
print("дно            : %s" % (tuple(bottom.shape),))

## 6 · Навчання: один протокол на всі заміри

Щоб порівняння були чесні, всі мережі до єдиної навчаються однаково: `AdamW`,
швидкість навчання 0.005, 22 епохи, батч 16, і в кожному кроці береться випадковий
шматок 32 × 32 з тренувальної сцени. Навчання на шматках — не наша вигадка, а
рішення з оригінальної статті про U-Net: мережа повністю згорткова, тож навчена
на шматках вона працює на цілому зображенні без жодних змін. Нам це ще й економить
учетверо часу.

Втрата — сума двох доданків. Перший — крос-ентропія з **вагами класів**: фону
дається вага 0.25, предмету — 1.0. Другий доданок — `Dice`, метрика з теми 29,
яка сама по собі не залежить від того, скільки на полотні фону. Разом вони
тримають рідкісний клас: без них частина зерен просто перестає передбачати
предмет і завмирає в тому самому стані, що й «усе фон».

In [ ]:
CROP = 32
CLASS_WEIGHTS = torch.tensor([0.25, 1.0])
EPOCHS, BATCH, LR = 8, 8, 3e-3
SEEDS = (0, 1, 2)


def dice_loss(logits, target, n_classes=2, smooth=1.0):
    """1 − Dice, усереднений по класах. Dice не залежить від частки фону."""
    probability = F.softmax(logits, 1)
    one_hot = F.one_hot(target, n_classes).permute(0, 3, 1, 2).float()
    over = (0, 2, 3)
    overlap = (probability * one_hot).sum(over)
    total = probability.sum(over) + one_hot.sum(over)
    return 1.0 - ((2 * overlap + smooth) / (total + smooth)).mean()


def train_model(model, images, masks, epochs=EPOCHS, seed=0, loss_kind="ce+dice"):
    """Навчання на випадкових шматках 32 × 32. Повертає витрачений час."""
    torch.manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    generator = torch.Generator().manual_seed(seed)
    how_many = len(images)
    model.train()
    started = time.perf_counter()
    for _epoch in range(epochs):
        order = torch.randperm(how_many, generator=generator)
        for start in range(0, how_many, BATCH):
            chosen = order[start:start + BATCH]
            corners = torch.randint(0, SIZE - CROP + 1, (len(chosen), 2),
                                    generator=generator)
            image_batch, mask_batch = [], []
            for j in range(len(chosen)):
                k = int(chosen[j])
                row, col = int(corners[j, 0]), int(corners[j, 1])
                image_batch.append(images[k, :, row:row + CROP, col:col + CROP])
                mask_batch.append(masks[k, row:row + CROP, col:col + CROP])
            image_batch = torch.stack(image_batch)
            mask_batch = torch.stack(mask_batch)

            logits = model(image_batch)
            cross_entropy = F.cross_entropy(logits, mask_batch, weight=CLASS_WEIGHTS)
            if loss_kind == "ce":
                loss = cross_entropy
            elif loss_kind == "dice":
                loss = dice_loss(logits, mask_batch)
            else:
                loss = cross_entropy + dice_loss(logits, mask_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return time.perf_counter() - started


@torch.no_grad()
def predict(model, images, batch=32):
    """Передбачення на цілих сценах 64 × 64 — мережа згорткова, розмір їй байдужий."""
    model.eval()
    pieces = []
    for start in range(0, len(images), batch):
        pieces.append(model(images[start:start + batch]).argmax(1))
    return torch.cat(pieces)


def measure(train_pack=None, val_pack=None, seeds=SEEDS, loss_kind="ce+dice", **kw):
    """Навчає ту саму мережу на трьох зернах і повертає середні з розкидом."""
    train_x, train_y = train_pack[0], train_pack[1]
    val_x, val_y = val_pack[0], val_pack[1]
    val_band = boundary_band(val_y, 2)
    # смуга завширшки 3 пікселі вздовж краю зображення — знадобиться в розділі 14
    val_edge = torch.zeros_like(val_y, dtype=torch.bool)
    val_edge[:, :3, :] = True
    val_edge[:, -3:, :] = True
    val_edge[:, :, :3] = True
    val_edge[:, :, -3:] = True
    collected = {"miou": [], "band": [], "acc_band": [], "acc_edge": [], "seconds": [],
                 "iou_0": [], "iou_1": []}
    for seed in seeds:
        torch.manual_seed(seed)
        model = UNet(**kw)
        collected["seconds"].append(
            train_model(model, train_x, train_y, seed=seed, loss_kind=loss_kind))
        predicted = predict(model, val_x)
        collected["miou"].append(mean_iou(predicted, val_y))
        collected["band"].append(mean_iou(predicted, val_y, only_where=val_band))
        collected["acc_band"].append(pixel_accuracy(predicted, val_y, only_where=val_band))
        collected["acc_edge"].append(pixel_accuracy(predicted, val_y, only_where=val_edge))
        for c in range(2):
            collected["iou_%d" % c].append(class_iou(predicted, val_y, c))
    result = {}
    for key, values in collected.items():
        clean = [v for v in values if v == v]
        result[key] = (float(np.mean(clean)), float(np.std(clean))) if clean else (float("nan"), 0.0)
    result["params"] = count_params(UNet(**kw))
    return result


def show(label, result):
    print("%-26s пар %7d | mIoU %.4f ±%.3f | межа %.4f ±%.3f | точн. межі %.4f ±%.3f | %5.1f с"
          % (label, result["params"], result["miou"][0], result["miou"][1],
             result["band"][0], result["band"][1],
             result["acc_band"][0], result["acc_band"][1], result["seconds"][0]))


TRAIN = (train_images, train_masks)
VAL = (val_images, val_masks)
print("протокол: %d епох, батч %d, шматки %d × %d, lr %g, зерна %s"
      % (EPOCHS, BATCH, CROP, CROP, LR, SEEDS))

## 7 · Замір 1: порогова глибина

Головний замір теми. Та сама мережа, чотири глибини стискання, два режими —
зі скіпами й без, по три зерна кожна точка. Двадцять чотири навчання.

Питання не «чи допомагають скіпи» (це знає кожен), а **з якої глибини вони
починають вирішувати**.

In [ ]:
depth_table = {}
for depth in (1, 2, 3, 4):
    for skip in (True, False):
        key = (depth, skip)
        depth_table[key] = measure(TRAIN, VAL, depth=depth, use_skip=skip)
        show("глибина %d, %s" % (depth, "зі скіпами" if skip else "без скіпів"),
             depth_table[key])
    with_skip = depth_table[(depth, True)]
    without = depth_table[(depth, False)]
    gap = with_skip["miou"][0] - without["miou"][0]
    spread = max(with_skip["miou"][1], without["miou"][1])
    print("   → розрив %.4f при розкиді %.4f — %s"
          % (gap, spread, "різниця" if gap > spread else "у межах розкиду"))
    print()

In [ ]:
levels = [1, 2, 3, 4]
skip_mean = [depth_table[(d, True)]["miou"][0] for d in levels]
skip_err = [depth_table[(d, True)]["miou"][1] for d in levels]
plain_mean = [depth_table[(d, False)]["miou"][0] for d in levels]
plain_err = [depth_table[(d, False)]["miou"][1] for d in levels]

plt.figure(figsize=(7, 3.6))
plt.errorbar(levels, skip_mean, yerr=skip_err, marker="o", capsize=4, label="зі скіпами")
plt.errorbar(levels, plain_mean, yerr=plain_err, marker="s", capsize=4, label="без скіпів")
plt.xticks(levels)
plt.xlabel("рівнів стискання")
plt.ylabel("mIoU на перевірці")
plt.title("порогова глибина: де скіпи починають вирішувати")
plt.grid(alpha=.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8 · Замір 1б: те саме на межі

Тепер міряємо ті самі мережі, але тільки на смузі межі. Загальний `mIoU` бачить
різницю теж — але межа показує її сильніше, бо саме там її джерело.

In [ ]:
print("глибина  розрив за загальним mIoU   розрив на межі   у скільки разів більший")
for depth in (1, 2, 3, 4):
    gap_all = (depth_table[(depth, True)]["miou"][0]
               - depth_table[(depth, False)]["miou"][0])
    gap_band = (depth_table[(depth, True)]["band"][0]
                - depth_table[(depth, False)]["band"][0])
    ratio = gap_band / gap_all if abs(gap_all) > 1e-6 else float("nan")
    print("   %d            %+.4f                %+.4f            %.2f"
          % (depth, gap_all, gap_band, ratio))

## 9 · Скіпи роблять навчання передбачуваним

Окремий результат, який видно в тій самій таблиці: розкид по зернах. Мережа зі
скіпами дає майже однаковий результат із будь-якого початкового наближення, мережа
без скіпів — ні.

In [ ]:
print("глибина   розкид зі скіпами   розкид без скіпів   у скільки разів більший")
for depth in (1, 2, 3, 4):
    a = depth_table[(depth, True)]["miou"][1]
    b = depth_table[(depth, False)]["miou"][1]
    print("   %d           %.4f              %.4f              %s"
          % (depth, a, b, "%.1f" % (b / a) if a > 1e-6 else "—"))

## 10 · Замір 2: конкатенація проти додавання

Тема 14 дала залишкові скіпи: `x + F(x)`, тензори **додаються**. U-Net зшиває
канали: `cat([x, skip])`. Різниця не косметична — при додаванні декодер отримує
суму двох сигналів і мусить розділити їх сам; при конкатенації вони лежать в
окремих каналах, і перша ж згортка блоку може дати їм різні ваги.

Ціна конкатенації — удвічі більше вхідних каналів у кожному блоці декодера.

In [ ]:
concat_result = depth_table[(3, True)]
add_result = measure(TRAIN, VAL, depth=3, use_skip=True, merge="add")
show("конкатенація (глибина 3)", concat_result)
show("додавання (глибина 3)", add_result)
print()
print("параметрів більше на %.1f %%"
      % (100 * (concat_result["params"] / add_result["params"] - 1)))
print("mIoU  конкатенація %.4f ±%.3f   додавання %.4f ±%.3f   різниця %+.4f"
      % (concat_result["miou"][0], concat_result["miou"][1],
         add_result["miou"][0], add_result["miou"][1],
         concat_result["miou"][0] - add_result["miou"][0]))
print("межа  конкатенація %.4f            додавання %.4f            різниця %+.4f"
      % (concat_result["band"][0], add_result["band"][0],
         concat_result["band"][0] - add_result["band"][0]))
print("час   конкатенація %.1f с          додавання %.1f с"
      % (concat_result["seconds"][0], add_result["seconds"][0]))

## 11 · Замір 3: який саме скіп важить

Вимикаємо скіпи по одному. Очікування підказує, що найважливіший —
**найдрібніший**, тобто той, що йде з рівня найвищої роздільності: саме він несе
край предмета в повному розмірі. Перевіримо, а не припустимо.

In [ ]:
ablations = {
    "усі три скіпи": [True, True, True],
    "без найдрібнішого": [False, True, True],
    "без найглибшого": [True, True, False],
    "жодного скіпа": [False, False, False],
}
ablation_results = {}
for label, pattern in ablations.items():
    if pattern == [True, True, True]:
        ablation_results[label] = depth_table[(3, True)]
    elif pattern == [False, False, False]:
        ablation_results[label] = depth_table[(3, False)]
    else:
        ablation_results[label] = measure(TRAIN, VAL, depth=3, use_skip=pattern)
    show(label, ablation_results[label])

full = ablation_results["усі три скіпи"]["miou"][0]
print()
for label in ablations:
    print("%-20s втрата від повного набору %+.4f"
          % (label, ablation_results[label]["miou"][0] - full))

## 12 · Замір 4: що саме несе скіп

Візьмемо одну навчену мережу й подивимось на дві карти, які зустрічаються в
кожному блоці декодера: ту, що прийшла **скіпом** з енкодера, і ту, що прийшла
**знизу**, з глибших шарів. Порівняємо їх чотирма числами:

- **частка ненульових** — скільки клітинок узагалі щось кажуть;
- **дисперсія** — наскільки карта контрастна;
- **кореляція з межею** — чи спалахує карта саме там, де в масці кордон;
- **висока частота** — яка частка сигналу зникає після згладжування 3 × 3.
  Це і є «дрібні деталі»: у гладкій карті ця частка мала.

In [ ]:
torch.manual_seed(0)
probe = UNet(depth=3)
train_model(probe, train_images, train_masks, seed=0)
probe_pred = predict(probe, val_images)
print("мережа для розбору: mIoU %.4f, на межі %.4f"
      % (mean_iou(probe_pred, val_masks), mean_iou(probe_pred, val_masks, only_where=band)))

probe.eval()
with torch.no_grad():
    _, saved, from_below, bottom = probe(val_images[:48], return_feats=True)

thin_band = boundary_band(val_masks[:48], 1).float().unsqueeze(1)


def describe(tensor, label):
    nonzero = (tensor > 1e-6).float().mean().item()
    variance = tensor.var(dim=(2, 3)).mean().item()
    average_map = tensor.mean(1, keepdim=True)
    step = 64 // tensor.shape[-1]
    target = F.max_pool2d(thin_band, step) if step > 1 else thin_band
    a = average_map.flatten(1)
    b = target.flatten(1)
    a = a - a.mean(1, keepdim=True)
    b = b - b.mean(1, keepdim=True)
    correlation = ((a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-9)).mean().item()
    smoothed = F.avg_pool2d(average_map, 3, 1, 1)
    high_frequency = ((average_map - smoothed).abs().mean()
                      / (average_map.abs().mean() + 1e-9)).item()
    print("%-32s %2dx%-2d  ненульових %.3f  дисперсія %6.3f  межа %+.3f  висока частота %.3f"
          % (label, tensor.shape[-1], tensor.shape[-1], nonzero, variance,
             correlation, high_frequency))


for level in range(3):
    describe(saved[level], "скіп із рівня %d" % level)
print()
for step, level in enumerate((2, 1, 0)):
    describe(from_below[step], "знизу на рівень %d" % level)

## 13 · Замір 5: чи потрібна симетрія

В оригінальній U-Net енкодер і декодер дзеркальні: скільки каналів на спуску,
стільки й на підйомі. Перевіримо, чи це обовʼязково: зробимо декодер удвічі
тоншим і подивимось, що зміниться.

In [ ]:
thin_decoder = measure(TRAIN, VAL, depth=3, use_skip=True, dec_scale=0.5)
show("симетричний декодер", depth_table[(3, True)])
show("декодер удвічі тонший", thin_decoder)
print()
print("параметрів менше на %.1f %%, mIoU змінився на %+.4f при розкиді %.4f"
      % (100 * (1 - thin_decoder["params"] / depth_table[(3, True)]["params"]),
         thin_decoder["miou"][0] - depth_table[(3, True)]["miou"][0],
         max(thin_decoder["miou"][1], depth_table[(3, True)]["miou"][1])))

## 14 · Замір 6: дзеркальне доповнення на краях

Оригінальна U-Net працює **без** доповнення взагалі: кожна згортка 3 × 3 зʼїдає
по пікселю з кожного боку, карта меншає, і на виході маска менша за вхід. Щоб
дістати маску краю зображення, автори доповнювали вхід **дзеркально** — відбивали
смугу пікселів назовні.

Сучасні реалізації майже завжди беруть `padding='same'` із нулями. Порівняємо:
нулі проти дзеркала, і подивимось окремо на смугу шириною 3 пікселі вздовж краю
зображення — там, де різниця тільки й може бути.

In [ ]:
reflect_result = measure(TRAIN, VAL, depth=3, use_skip=True, pad_mode="reflect")
show("нулі (padding='same')", depth_table[(3, True)])
show("дзеркальне доповнення", reflect_result)

edge_share = 1 - (58 * 58) / (64 * 64)
print()
print("смуга краю завширшки 3 px накриває %.2f %% пікселів" % (100 * edge_share))
print("нулі      точність на краю %.4f ±%.4f"
      % depth_table[(3, True)]["acc_edge"])
print("дзеркало  точність на краю %.4f ±%.4f" % reflect_result["acc_edge"])
print("різниця   %+.4f при розкиді %.4f"
      % (reflect_result["acc_edge"][0] - depth_table[(3, True)]["acc_edge"][0],
         max(reflect_result["acc_edge"][1], depth_table[(3, True)]["acc_edge"][1])))

## 15 · Замір 7: кільце — структура, заради якої U-Net і придумали

U-Net народилась у біомедичних знімках, де корисні структури тонкі: мембрани,
судини, контури клітин. Наш аналог такої структури — **кільце**: коло з
вирізаною серединою, у якого вся корисна частина — стінка завтовшки кілька
пікселів.

Замінимо трикутник на кільце. Товщина стінки в кожного кільця своя — 1, 2 або 3
пікселі, — тож одна навчена мережа одразу міряється на всіх трьох товщинах, і
три точки коштують одне навчання замість трьох.

In [ ]:
RING_KINDS = (0, 1, 3)          # коло, квадрат, кільце
# в одному наборі зустрічаються кільця всіх трьох товщин — тоді одна навчена
# мережа міряється одразу на всіх, і три товщини коштують не три навчання, а одне
ring_train_raw = make_dataset(256, seed=42, kinds=RING_KINDS, ring_wall="mixed")
ring_val_raw = make_dataset(96, seed=7, kinds=RING_KINDS, ring_wall="mixed")
ring_train = (ring_train_raw[0], (ring_train_raw[1] > 0).long())
ring_val = (ring_val_raw[0], (ring_val_raw[1] > 0).long())
ring_val_walls = ring_val_raw[2]

print("кільця займають %.2f %% пікселів перевірного набору"
      % (100 * (ring_val_raw[1] == 3).float().mean().item()))
for wall in (1, 2, 3):
    print("   зі стінкою %d px: %.2f %% пікселів"
          % (wall, 100 * (ring_val_walls == wall).float().mean().item()))


def ring_iou_by_wall(predicted, truth_masks, wall_map, wall):
    """IoU кільця зі стінкою заданої товщини.

    Обʼєднання беремо не по всьому полотну, а тільки в околі цих кілець —
    інакше в знаменник потрапили б кільця іншої товщини.
    """
    truth = wall_map == wall
    if truth.sum() == 0:
        return float("nan")
    # околиця: розширюємо істинні пікселі на 3 у всі боки
    near = F.max_pool2d(truth.unsqueeze(1).float(), 7, 1, 3).squeeze(1) > 0
    guess = (predicted == 1) & near
    union = (truth | guess).sum().item()
    return (truth & guess).sum().item() / union


def measure_ring(**kw):
    """Те саме, що measure, але з IoU кільця окремо по товщинах стінки."""
    per_wall = {1: [], 2: [], 3: []}
    whole = []
    for seed in SEEDS:
        torch.manual_seed(seed)
        model = UNet(**kw)
        train_model(model, ring_train[0], ring_train[1], seed=seed)
        predicted = predict(model, ring_val[0])
        whole.append(class_iou(predicted, ring_val[1], 1))
        for wall in (1, 2, 3):
            per_wall[wall].append(
                ring_iou_by_wall(predicted, ring_val[1], ring_val_walls, wall))
    result = {"iou_ring": (float(np.mean(whole)), float(np.std(whole)))}
    for wall in (1, 2, 3):
        result["wall_%d" % wall] = (float(np.mean(per_wall[wall])),
                                    float(np.std(per_wall[wall])))
    return result


ring_with = measure_ring(depth=3, use_skip=True)
ring_without = measure_ring(depth=3, use_skip=False)

print()
print("IoU предмета цілком:  зі скіпами %.4f ±%.3f   без скіпів %.4f ±%.3f"
      % (ring_with["iou_ring"][0], ring_with["iou_ring"][1],
         ring_without["iou_ring"][0], ring_without["iou_ring"][1]))
print()
print("товщина стінки   зі скіпами        без скіпів        розрив")
for wall in (1, 2, 3):
    a = ring_with["wall_%d" % wall]
    b = ring_without["wall_%d" % wall]
    print("     %d px        %.4f ±%.3f     %.4f ±%.3f     %+.4f"
          % (wall, a[0], a[1], b[0], b[1], a[0] - b[0]))

### Втрата й глибина на тонкій структурі

Тема 29 міряла втрати на звичайних фігурах. Тонка структура — інша річ: пікселів
у неї мало, і крос-ентропія, навіть із вагами класів, легше миряться з тим, щоб
її втратити. Порівняємо чисту крос-ентропію з сумою «крос-ентропія плюс Dice» на
двох глибинах.

In [ ]:
print("глибина  втрата      IoU кільця        стінка 1 px      стінка 3 px")
loss_table = {}
for depth in (2, 3):
    for loss_kind in ("ce", "ce+dice"):
        if depth == 3 and loss_kind == "ce+dice":
            result = ring_with
        else:
            per_wall = {1: [], 3: []}
            whole = []
            for seed in SEEDS:
                torch.manual_seed(seed)
                model = UNet(depth=depth, use_skip=True)
                train_model(model, ring_train[0], ring_train[1], seed=seed,
                            loss_kind=loss_kind)
                predicted = predict(model, ring_val[0])
                whole.append(class_iou(predicted, ring_val[1], 1))
                for wall in (1, 3):
                    per_wall[wall].append(
                        ring_iou_by_wall(predicted, ring_val[1], ring_val_walls, wall))
            result = {"iou_ring": (float(np.mean(whole)), float(np.std(whole)))}
            for wall in (1, 3):
                result["wall_%d" % wall] = (float(np.mean(per_wall[wall])),
                                            float(np.std(per_wall[wall])))
        loss_table[(depth, loss_kind)] = result
        print("   %d     %-10s  %.4f ±%.3f     %.4f          %.4f"
              % (depth, loss_kind, result["iou_ring"][0], result["iou_ring"][1],
                 result["wall_1"][0], result["wall_3"][0]))

## 16 · Що вийшло

Зберемо головне в одну таблицю.

In [ ]:
print("=" * 74)
print("ПОРОГОВА ГЛИБИНА")
print("=" * 74)
print("глибина   зі скіпами          без скіпів          розрив   розкид   висновок")
threshold = None
for depth in (1, 2, 3, 4):
    a = depth_table[(depth, True)]
    b = depth_table[(depth, False)]
    gap = a["miou"][0] - b["miou"][0]
    spread = max(a["miou"][1], b["miou"][1])
    decisive = gap > spread
    if decisive and threshold is None:
        threshold = depth
    print("   %d      %.4f ±%.3f      %.4f ±%.3f      %+.4f   %.4f   %s"
          % (depth, a["miou"][0], a["miou"][1], b["miou"][0], b["miou"][1],
             gap, spread, "скіпи вирішують" if decisive else "у межах розкиду"))
print()
if threshold is None:
    print("порогової глибини не знайдено: на жодній глибині розрив не перевищив розкид")
else:
    print("ПОРОГОВА ГЛИБИНА = %d: з неї розрив стає більшим за розкид по зернах" % threshold)
print()
print("витрачено на весь зошит: %.0f с" % (time.perf_counter() - NOTEBOOK_STARTED))

## Завдання

**🟢 Рівень 1.** Поміряй, як ширина смуги межі впливає на розрив між мережею зі
скіпами й без. Візьми `w = 1, 2, 3, 5`, для кожного порахуй частку пікселів у
смузі й `mIoU` обох мереж глибини 3 на ній. **Зроблено, якщо** є таблиця з
чотирьох рядків і сказано, у який бік змінюється розрив, коли смуга вужчає.

**🟡 Рівень 2.** Підміни те, що йде скіпом, на етапі передбачення: нулі, розмита
версія, перемішана між прикладами. Для кожної підміни порахуй `mIoU` загалом і
на межі. **Зроблено, якщо** з трьох чисел видно, чого саме бракує декодерові —
роздільності чи семантики.

**🔴 Рівень 3.** Знайди порогову глибину на **своїх** даних: візьми будь-яку
задачу з масками, повтори розділ 7 повністю (чотири глибини, два режими, три
зерна) і назви глибину, з якої скіпи стають вирішальними. Відповідь «різниці не
виявлено» теж приймається — але тільки з таблицею, де розрив на кожній глибині
менший за розкид. **Зроблено, якщо** є таблиця 8 × 3 і одне речення про те, чим
твої дані відрізняються від наших сцен.

Розгорнуті умови — у [homework.html](homework.html).